# Chương 1: Đánh giá Hiệu năng thuật toán FP-Growth
Notebook này cấu hình các tham số và tự động chạy cả thuật toán tự phát triển (Julia) lẫn phần mềm tham chiếu (SPMF Java) qua nhiều mức `min_sup` khác nhau để thu thập dữ liệu Thời gian & Bộ nhớ.

In [17]:
# 1. IMPORT CÁC THƯ VIỆN CẦN THIẾT
include("../src/FPGrowthProject.jl")
using .FPGrowthProject
import Pkg

# Tự động cài đặt DataFrames và CSV nếu máy chưa có
if !haskey(Pkg.project().dependencies, "DataFrames") || !haskey(Pkg.project().dependencies, "CSV")
    Pkg.add("DataFrames")
    Pkg.add("CSV")
end

using CSV
using DataFrames
using Random
Random.seed!(42)
println("Đã cấu hình Random Seed để cố định kết quả!")


Đã cấu hình Random Seed để cố định kết quả!


In [18]:
# 2. CẤU HÌNH (CONFIG)
CONFIG = Dict(
    "data_path" => "../data/benchmark/accidents.dat/accidents.dat",
    "spmf_path" => "../spmf.jar",
    "java_path" => "C:/Program Files/Microsoft/jdk-21.0.10.7-hotspot/bin/java.exe", # Bạn có thể đổi thành "java" nếu PATH đã nhận chuẩn
    "output_julia" => "../results/output_julia.txt",
    "output_spmf" => "../results/output_spmf.txt",
    "metrics_file" => "../results/results_metrics.csv",
    "min_sups" => [0.8, 0.7, 0.6, 0.5, 0.4] # Các ngưỡng test
)

# Tạo thư mục results nếu chưa có
if !isdir("../results")
    mkdir("../results")
end


In [19]:
# 3. HÀM CHẠY EXPERIMENT & PARSE LOG
function run_experiment(config)
    println("--- BẮT ĐẦU ĐỌC DỮ LIỆU ---")
    transactions = FPGrowthProject.read_spmf(config["data_path"])
    total_txs = length(transactions)
    println("Tổng số giao dịch: ", total_txs)
    
    # Tải SPMF nếu chưa có
    if !isfile(config["spmf_path"])
        download("https://www.philippe-fournier-viger.com/spmf/spmf.jar", config["spmf_path"])
    end

    results_df = DataFrame(MinSup=Float64[], JuliaTime=Float64[], JuliaMemMB=Float64[], SPMFTime=Float64[], SPMFMemMB=Float64[])

    for min_sup_ratio in config["min_sups"]
        println("\n========================================")
        println("Chạy với min_sup = ", min_sup_ratio * 100, "%")
        
        # 1. Chạy Julia
        min_sup_abs = ceil(Int, min_sup_ratio * total_txs)
        println("[Julia] Bắt đầu...")
        # Đo bộ nhớ và thời gian thủ công
        gc_stats_before = Base.gc_num()
        time_before = time_ns()
        allocated_before = Base.gc_num().allocd
        
        julia_result = FPGrowthProject.fpgrowth(transactions, min_sup_abs)
        
        time_after = time_ns()
        allocated_after = Base.gc_num().allocd
        
        julia_time = (time_after - time_before) / 1e9 # Seconds
        julia_mem_mb = (allocated_after - allocated_before) / (1024^2) # MB
        println("[Julia] Xong! Thời gian: ", round(julia_time, digits=3), "s | RAM: ", round(julia_mem_mb, digits=2), " MB")
        
        # Lưu file output của Julia (Chỉ lưu ở mức min_sup thấp nhất để check correctness)
        if min_sup_ratio == minimum(config["min_sups"])
            FPGrowthProject.write_spmf(config["output_julia"], julia_result)
        end
        
        # 2. Chạy SPMF Java
        println("[SPMF] Bắt đầu...")
        cmd = `$(config["java_path"]) -jar $(config["spmf_path"]) run FPGrowth_itemsets $(config["data_path"]) $(config["output_spmf"]) $min_sup_ratio`
        
        # Đọc log output của Java để lấy RAM và Time
        io = IOBuffer()
        run(pipeline(cmd, stdout=io))
        spmf_output = String(take!(io))
        
        # Regex bóc tách thời gian và RAM từ Output SPMF
        spmf_time = 0.0
        spmf_mem_mb = 0.0
        
        for line in split(spmf_output, "\n")
            if occursin("Total time", line)
                m = match(r"~ (\d+) ms", line)
                if m !== nothing spmf_time = parse(Float64, m.captures[1]) / 1000.0 end
            elseif occursin("Max memory usage", line)
                m = match(r": ([\d\.]+) mb", line)
                if m !== nothing spmf_mem_mb = parse(Float64, m.captures[1]) end
            end
        end
        println("[SPMF] Xong! Thời gian: ", round(spmf_time, digits=3), "s | RAM: ", round(spmf_mem_mb, digits=2), " MB")

        # Lưu số liệu vào DataFrame
        push!(results_df, (min_sup_ratio, julia_time, julia_mem_mb, spmf_time, spmf_mem_mb))
    end
    
    # Ghi ra CSV
    CSV.write(config["metrics_file"], results_df)
    println("\nĐã xuất kết quả ra file: ", config["metrics_file"])
    return results_df
end



run_experiment (generic function with 1 method)

In [20]:
# 4. CHẠY!
df = run_experiment(CONFIG)
display(df)


--- BẮT ĐẦU ĐỌC DỮ LIỆU ---
Tổng số giao dịch: 340183

Chạy với min_sup = 80.0%
[Julia] Bắt đầu...
[Julia] Xong! Thời gian: 0.745s | RAM: 41.65 MB
[SPMF] Bắt đầu...
[SPMF] Xong! Thời gian: 1.358s | RAM: 15.82 MB

Chạy với min_sup = 70.0%
[Julia] Bắt đầu...
[Julia] Xong! Thời gian: 0.607s | RAM: 41.05 MB
[SPMF] Bắt đầu...
[SPMF] Xong! Thời gian: 1.399s | RAM: 93.47 MB

Chạy với min_sup = 60.0%
[Julia] Bắt đầu...
[Julia] Xong! Thời gian: 1.073s | RAM: -16.3 MB
[SPMF] Bắt đầu...
[SPMF] Xong! Thời gian: 1.813s | RAM: 120.59 MB

Chạy với min_sup = 50.0%
[Julia] Bắt đầu...
[Julia] Xong! Thời gian: 1.596s | RAM: 29.04 MB
[SPMF] Bắt đầu...
[SPMF] Xong! Thời gian: 1.929s | RAM: 41.92 MB

Chạy với min_sup = 40.0%
[Julia] Bắt đầu...
[Julia] Xong! Thời gian: 3.353s | RAM: 1.59 MB
[SPMF] Bắt đầu...
[SPMF] Xong! Thời gian: 2.56s | RAM: 287.45 MB

Đã xuất kết quả ra file: ../results/results_metrics.csv


Row,MinSup,JuliaTime,JuliaMemMB,SPMFTime,SPMFMemMB
,Float64,Float64,Float64,Float64,Float64
1,0.8,0.745145,41.6507,1.358,15.8225
2,0.7,0.607357,41.0516,1.399,93.4665
3,0.6,1.07318,-16.2982,1.813,120.589
4,0.5,1.5961,29.041,1.929,41.922
5,0.4,3.35315,1.58558,2.56,287.446
